# 第9章: 事前学習済み言語モデル（BERT型）

本章では、BERT型の事前学習済みモデルを利用して、マスク単語の予測や文ベクトルの計算、評判分析器（ポジネガ分類器）の構築に取り組む。

## 80. トークン化

"The movie was full of incomprehensibilities."という文をトークンに分解し、トークン列を表示せよ。

In [1]:
!pip install tiktoken

In [3]:
import tiktoken

text_to_tokenize = "The movie was full of incomprehensibilities."

encoding = tiktoken.get_encoding("cl100k_base")
tokens = encoding.encode(text_to_tokenize)

# 各トークンIDを文字列にデコードして表示
token_strings = [encoding.decode([t]) for t in tokens]

print(f"トークン列: {tokens}")
print(f"トークン文字列: {token_strings}")
print(f"トークン数: {len(tokens)}")

トークン列: [791, 5818, 574, 2539, 315, 53990, 31882, 729, 13757, 13]
トークン文字列: ['The', ' movie', ' was', ' full', ' of', ' incom', 'preh', 'ens', 'ibilities', '.']
トークン数: 10


## 81. マスクの予測

"The movie was full of [MASK]."の"[MASK]"を埋めるのに最も適切なトークンを求めよ。

In [4]:
!pip install transformers torch

In [7]:
from transformers import pipeline

# マスク予測用のパイプラインを初期化 (英語の基本モデルを使用)
mask_filler = pipeline("fill-mask", model="bert-base-uncased")

# 予測したい文
text = "The movie was full of [MASK]."

# 予測の実行
results = mask_filler(text)

# 結果の表示
print(f"入力文: {text}\n")
for result in results:
    print(f"スコア: {result['score']:.4f}, トークン: {result['token_str']}, 文: {result['sequence']}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

入力文: The movie was full of [MASK].

スコア: 0.1071, トークン: fun, 文: the movie was full of fun.
スコア: 0.0663, トークン: surprises, 文: the movie was full of surprises.
スコア: 0.0447, トークン: drama, 文: the movie was full of drama.
スコア: 0.0272, トークン: stars, 文: the movie was full of stars.
スコア: 0.0254, トークン: laughs, 文: the movie was full of laughs.


## 82. マスクのtop-k予測

"The movie was full of [MASK]."の"[MASK]"に埋めるのに適切なトークン上位10個と、その確率（尤度）を求めよ。

In [8]:
from transformers import pipeline

# マスク予測用のパイプラインを初期化
mask_filler = pipeline("fill-mask", model="bert-base-uncased")

# 予測したい文
text = "The movie was full of [MASK]."

# 上位10個の予測を取得
results = mask_filler(text, top_k=10)

# 結果の表示
print(f"入力文: {text}\n")
for i, result in enumerate(results, 1):
    print(f"{i:2}: スコア {result['score']:.4f} | トークン: {result['token_str']}")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


入力文: The movie was full of [MASK].

 1: スコア 0.1071 | トークン: fun
 2: スコア 0.0663 | トークン: surprises
 3: スコア 0.0447 | トークン: drama
 4: スコア 0.0272 | トークン: stars
 5: スコア 0.0254 | トークン: laughs
 6: スコア 0.0195 | トークン: action
 7: スコア 0.0190 | トークン: excitement
 8: スコア 0.0183 | トークン: people
 9: スコア 0.0150 | トークン: tension
10: スコア 0.0146 | トークン: music


## 83. CLSトークンによる文ベクトル

以下の文の全ての組み合わせに対して、最終層の[CLS]トークンの埋め込みベクトルを用いてコサイン類似度を求めよ。

- "The movie was full of fun."
- "The movie was full of excitement."
- "The movie was full of crap."
- "The movie was full of rubbish."


### [CLS]トークンの抽出例

各文章の先頭にある `[CLS]` トークン（ID: 101）に対応する、最終層の隠れ状態（Hidden State）を取得します。

In [9]:
import torch
from transformers import BertTokenizer, BertModel
from sklearn.metrics.pairwise import cosine_similarity

# モデルとトークナイザーの準備
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)

sentences = [
    "The movie was full of fun.",
    "The movie was full of excitement.",
    "The movie was full of crap.",
    "The movie was full of rubbish."
]

# [CLS] トークンのベクトルを取得する関数
def get_cls_embedding(text):
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    # outputs.last_hidden_state の形状は [バッチサイズ, 系列長, 隠れ層の次元数]
    # [CLS]は先頭（インデックス0）に存在します
    cls_vec = outputs.last_hidden_state[0, 0, :]
    return cls_vec

# 各文のベクトルを計算
embeddings = [get_cls_embedding(s) for s in sentences]

# 類似度の計算と表示
print("--- コサイン類似度 ([CLS]トークン) ---")
for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        sim = cosine_similarity(embeddings[i].reshape(1, -1), embeddings[j].reshape(1, -1))[0][0]
        print(f"{sentences[i]} vs {sentences[j]}: {sim:.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--- コサイン類似度 ([CLS]トークン) ---
The movie was full of fun. vs The movie was full of excitement.: 0.9881
The movie was full of fun. vs The movie was full of crap.: 0.9558
The movie was full of fun. vs The movie was full of rubbish.: 0.9475
The movie was full of excitement. vs The movie was full of crap.: 0.9541
The movie was full of excitement. vs The movie was full of rubbish.: 0.9487
The movie was full of crap. vs The movie was full of rubbish.: 0.9807


## 84. 平均による文ベクトル

以下の文の全ての組み合わせに対して、最終層の埋め込みベクトルの平均を用いてコサイン類似度を求めよ。

- "The movie was full of fun."
- "The movie was full of excitement."
- "The movie was full of crap."
- "The movie was full of rubbish."

## 85. データセットの準備

[General Language Understanding Evaluation (GLUE)](https://gluebenchmark.com/) ベンチマークで配布されている[Stanford Sentiment Treebank (SST)](https://dl.fbaipublicfiles.com/glue/data/SST-2.zip) から訓練セット（train.tsv）と開発セット（dev.tsv）のテキストと極性ラベルと読み込み、さらに全てのテキストはトークン列に変換せよ。

## 86. ミニバッチの作成

85で読み込んだ訓練データの一部（例えば冒頭の4事例）に対して、パディングなどの処理を行い、トークン列の長さを揃えてミニバッチを構成せよ。

## 87. ファインチューニング

訓練セットを用い、事前学習済みモデルを極性分析タスク向けにファインチューニングせよ。検証セット上でファインチューニングされたモデルの正解率を計測せよ。

## 88. 極性分析

問題87でファインチューニングされたモデルを用いて、以下の文の極性を予測せよ。

- "The movie was full of incomprehensibilities."
- "The movie was full of fun."
- "The movie was full of excitement."
- "The movie was full of crap."
- "The movie was full of rubbish."


## 89. アーキテクチャの変更

問題87とは異なるアーキテクチャ（例えば[CLS]トークンを用いるか、各トークンの最大値プーリングを用いるなど）の分類モデルを設計し、事前学習済みモデルを極性分析タスク向けにファインチューニングせよ。検証セット上でファインチューニングされたモデルの正解率を計測せよ。